### evaluation metric : ROC-AUC
* ROC곡선과 이에 기반한 AUC 스코어는 이진 분류의 예측 성능 측정에서 중요하게 사용되는 수치이다.
	* ROC곡선은 이진 분류 모델의 예측 성능을 판단하는 중요한 지표이다.
	* ROC곡선은 FPR이 변할 때, TPR이 어떻게 변하는지를 나타내는 곡선이다.
		* TPR(재현율, 민감도) = $\frac{TP}{TP + FN}$
		* TNR(특이성) = $\frac{TN}{TN + FP}$
		* FPR = 1 - TNR
	* 임계값을 [0, 1] 범위에서 변화시키면 그에 따라 FPR값이 0에서부터 1까지 점차 증가한다. 이를 X축으로 두고, Y축에는 TPR 값을 두고 그린 그래프가 ROC이다.
	* 그래프가 왼쪽 상단에 붙어있으면 성능이 좋은거고, 가운데 직선 가까이 있으면 분류 성능이 좋지 않은 것 이다.

* AUC(Area Under Curve)
	* 분류의 성능 지표로 사용되는 것은 ROC 곡선 면적에 기반한 AUC값이다.
	* AUC 지표가 0.5라면, 랜덤 찍기와 비슷한 수준이고, 1에 가까울 수록 좋은 수치를 의미한다.

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Null 처리 함수
def fillna(df):
	df['Age'] = df['Age'].fillna(df['Age'].mean())
	df['Cabin'] = df['Cabin'].fillna('N')
	df['Embarked'] = df['Embarked'].fillna('N')
	df['Fare'] = df['Fare'].fillna(0)
	return df

# 머신러닝 알고리즘에 불필요한 피처 제거
def drop_features(df):
	df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)
	return df

# 레이블 인코딩 수행.
def format_features(df):
	df['Cabin'] = df['Cabin'].str[:1]
	features = ['Cabin', 'Sex', 'Embarked']
	for feature in features:
		le = LabelEncoder()
		le = le.fit(df[feature])
		df[feature] = le.transform(df[feature])
	return df

# 앞에서 설정한 데이터 전처리 함수 호출
def transform_features(df):
	df = fillna(df)
	df = drop_features(df)
	df = format_features(df)
	return df

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

titanic_df = pd.read_csv('titanic_train.csv')
y_titanic_df = titanic_df['Survived']
X_titanic_df = titanic_df.drop('Survived', axis=1)
X_titanic_df = transform_features(X_titanic_df)
X_titanic_df['Age'].fillna(X_titanic_df['Age'].mean(), inplace=True)
X_train, X_test, y_train, y_test = train_test_split(X_titanic_df, y_titanic_df, test_size=0.2, random_state=11)

lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train, y_train)


[0.94 0.73 0.62 0.52 0.44 0.28 0.15 0.14 0.13 0.12]
[0.    0.008 0.025 0.076 0.127 0.254 0.576 0.61  0.746 0.847]
[0.016 0.492 0.705 0.738 0.803 0.885 0.902 0.951 0.967 1.   ]


/var/folders/c1/h7bpvp6j477dywpxvwgcb5qw0000gn/T/ipykernel_41949/2603372980.py:45: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  X_titanic_df['Age'].fillna(X_titanic_df['Age'].mean(), inplace=True)


In [2]:
import numpy as np
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score

pred_proba_class1 = lr_clf.predict_proba(X_test)[:, 1]

fprs, tprs, thresholds = roc_curve(y_test, pred_proba_class1)
index = np.arange(1, thresholds.shape[0], 5)
print(np.round(thresholds[index], 2))
print(np.round(fprs[index], 3))
print(np.round(tprs[index], 3))

roc_score = roc_auc_score(y_test, pred_proba_class1)
print('{0:.4f}'.format(roc_score))

[0.94 0.73 0.62 0.52 0.44 0.28 0.15 0.14 0.13 0.12]
[0.    0.008 0.025 0.076 0.127 0.254 0.576 0.61  0.746 0.847]
[0.016 0.492 0.705 0.738 0.803 0.885 0.902 0.951 0.967 1.   ]
0.8987
